# Tutorial: Supermix Kaggle Current Training

Audience:
- You, when you want the current Qwen Supermix training path on Kaggle without switching architectures or rewriting the trainer.

Goals:
- Reuse the same `source/qwen_supermix_pipeline.py` pipeline that the repo uses now.
- Keep the same Qwen LoRA training architecture, grouped eval splitting, true SFT packing, preference mining, and newer research knobs.
- Use Kaggle input datasets for source code, training JSONL files, optional teacher files, and optional warm-start artifacts.
- Default to settings that are safer on Kaggle GPU runtimes such as T4, P100, and L4.

Important:
- This notebook stays on the Qwen LoRA pipeline.
- It does not switch to the legacy classifier path in `finetune_chat.py` or the retrieval/database path in `llm_database.py`.
- Kaggle storage is different from Colab Drive: `/kaggle/input` is read-only, `/kaggle/working` is writable, and you must export a bundle if you want to carry checkpoints into another session.


## Outline

1. Verify the Kaggle GPU runtime.
2. Materialize the training workspace from a Kaggle dataset attachment or Git clone fallback.
3. Optionally audit the attached training dataset bundle with SQLite.
4. Build the current Qwen training command.
5. Launch training.
6. Reattach to progress and package a Kaggle resume bundle.


In [ ]:
from __future__ import annotations

import platform
import shutil
import subprocess
import sys
import urllib.request
import zipfile

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
if shutil.which('nvidia-smi') is None:
    raise RuntimeError(
        'Kaggle is not attached to a GPU runtime. In Notebook Settings, enable Accelerator = GPU, then rerun from the top.'
    )
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
import os
from pathlib import Path

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
KAGGLE_WORKING_ROOT = Path('/kaggle/working/supermix_kaggle_current')

REQUESTED_SOURCE_DATASET_SLUG = 'supermix-source'
REQUESTED_TRAINING_DATASET_SLUG = 'supermix-datasets'
REQUESTED_TEACHER_DATASET_SLUG = 'supermix-runtime-python'
REQUESTED_WARM_START_DATASET_SLUG = 'supermix-warm-start'

TRAINING_DATASET_HINTS = (
    'conversation_data.quality_anchor_v2.jsonl',
    'conversation_data.coding_knowledge_2026_02_19.jsonl',
    'conversation_data.world_events_2026_02_19.jsonl',
    'conversation_data.supermix_plus_v27_500k.jsonl',
    'conversation_data.mega_reasoning_creative_v25_75582.jsonl',
    'conversation_data.mega_creative_250k_v2.jsonl',
)


def list_input_mounts() -> list[Path]:
    if not KAGGLE_INPUT_ROOT.exists():
        return []
    return sorted((path for path in KAGGLE_INPUT_ROOT.iterdir() if path.is_dir()), key=lambda path: path.name.lower())


def dataset_hit_count(root: Path, names: tuple[str, ...]) -> int:
    hits = 0
    for name in names:
        direct_candidates = (
            root / name,
            root / 'source' / name,
            root / 'datasets' / name,
            root / 'adapter' / name,
            root / 'runtime_python' / name,
        )
        if any(candidate.exists() for candidate in direct_candidates):
            hits += 1
            continue
        if any(hit.is_file() for hit in root.rglob(name)):
            hits += 1
    return hits


def resolve_input_dir(preferred_slug: str, required_names: tuple[str, ...]) -> Path:
    mounts = list_input_mounts()
    preferred = KAGGLE_INPUT_ROOT / preferred_slug
    if preferred.exists():
        preferred_hits = dataset_hit_count(preferred, required_names)
        if preferred_hits == len(required_names) or preferred_hits > 0 or not required_names:
            return preferred

    best_candidate = preferred
    best_hits = -1
    for candidate in mounts:
        hits = dataset_hit_count(candidate, required_names)
        if hits > best_hits:
            best_candidate = candidate
            best_hits = hits
        if required_names and hits == len(required_names):
            return candidate

    if best_hits > 0:
        return best_candidate
    return preferred


MOUNTED_INPUTS = [path.name for path in list_input_mounts()]

SOURCE_INPUT_DIR = resolve_input_dir(REQUESTED_SOURCE_DATASET_SLUG, ('qwen_supermix_pipeline.py',))
TRAINING_INPUT_DIR = resolve_input_dir(REQUESTED_TRAINING_DATASET_SLUG, TRAINING_DATASET_HINTS)
TEACHER_INPUT_DIR = resolve_input_dir(
    REQUESTED_TEACHER_DATASET_SLUG,
    ('chat_model_meta_supermix_v27_500k.json', 'champion_model_chat_supermix_v27_500k_ft.pth'),
)
WARM_START_INPUT_DIR = resolve_input_dir(
    REQUESTED_WARM_START_DATASET_SLUG,
    ('latest_adapter_checkpoint.txt', 'adapter_config.json', 'benchmark_results.json'),
)

SOURCE_DATASET_SLUG = SOURCE_INPUT_DIR.name if SOURCE_INPUT_DIR.exists() else REQUESTED_SOURCE_DATASET_SLUG
TRAINING_DATASET_SLUG = TRAINING_INPUT_DIR.name if TRAINING_INPUT_DIR.exists() else REQUESTED_TRAINING_DATASET_SLUG
TEACHER_DATASET_SLUG = TEACHER_INPUT_DIR.name if TEACHER_INPUT_DIR.exists() else REQUESTED_TEACHER_DATASET_SLUG
WARM_START_DATASET_SLUG = WARM_START_INPUT_DIR.name if WARM_START_INPUT_DIR.exists() else REQUESTED_WARM_START_DATASET_SLUG

HF_HOME = KAGGLE_WORKING_ROOT / 'hf_cache'
LOG_DIR = KAGGLE_WORKING_ROOT / 'logs'
ARTIFACT_ROOT = KAGGLE_WORKING_ROOT / 'artifacts'
OUTPUT_DIR = ARTIFACT_ROOT / 'qwen_supermix_enhanced_v28_kaggle_current'
RESUME_WARM_START_DIR = ARTIFACT_ROOT / 'qwen_supermix_kaggle_warm_start'
STATE_PATH = KAGGLE_WORKING_ROOT / 'last_training_launch_current.json'
PROCESS_PID_PATH = KAGGLE_WORKING_ROOT / 'train_process.pid'
SQLITE_MANIFEST_PATH = KAGGLE_WORKING_ROOT / 'dataset_manifest.sqlite3'
WORKSPACE_DIR = KAGGLE_WORKING_ROOT / 'Supermix_29'
REPO_URL = 'https://github.com/kai9987kai/Supermix_29.git'
BRANCH = 'main'
ALLOW_GIT_CLONE_FALLBACK = True
TRY_REPO_LFS_ASSETS = False
REPO_WARM_START_PATH: str | None = None
REPO_BUNDLED_WARM_START_PATH: str | None = None

HF_BASE_MODEL_REPO = 'Qwen/Qwen2.5-0.5B-Instruct'
HF_BASE_MODEL_REVISION = '7ae557604adf67be50417f59c2c2f167def9a775'
BASE_MODEL_DIR = KAGGLE_WORKING_ROOT / 'base_models' / f'qwen2_5_0_5b_instruct_{HF_BASE_MODEL_REVISION}'
BASE_MODEL = str(BASE_MODEL_DIR)

DEVICE = 'cuda'
SAVE_EVERY_STEPS = 80
TRAIN_PROFILE = 'fast_free_t4'  # fast_free_t4, current_kaggle, or parity_v28
GPU_PROFILE = 'auto'  # auto, t4, p100, l4, a100, or generic
LAUNCH_MODE = 'background'  # background or stream
FORCE_RESTART = False
PACKAGE_RESUME_BUNDLE = False
RUN_BENCHMARK_AFTER_TRAIN = False
BENCHMARK_EVAL_LIMIT = 256
ENABLE_SQLITE_DATASET_MANIFEST = False
ALLOW_BUNDLED_WARM_START_FALLBACK = False
EXTRA_ARGS: list[str] = []
PINNED_VERSIONS: dict[str, str] | None = None

for path in (KAGGLE_WORKING_ROOT, HF_HOME, LOG_DIR, ARTIFACT_ROOT, OUTPUT_DIR, BASE_MODEL_DIR.parent):
    path.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME'] = str(HF_HOME)
os.environ['TRANSFORMERS_CACHE'] = str(HF_HOME / 'transformers')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(HF_HOME / 'hub')
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['PYTHONHASHSEED'] = '48'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ.pop('HF_HUB_OFFLINE', None)
os.environ.pop('TRANSFORMERS_OFFLINE', None)

if TRAIN_PROFILE in {'fast_free_t4', 'current_kaggle', 'parity_v28'}:
    EXTRA_ARGS = [
        '--strict_determinism',
        '--disable_tf32',
        '--matmul_precision', 'highest',
    ]
else:
    raise ValueError(f'Unsupported TRAIN_PROFILE: {TRAIN_PROFILE}')

{
    'MOUNTED_INPUTS': MOUNTED_INPUTS,
    'REQUESTED_SOURCE_DATASET_SLUG': REQUESTED_SOURCE_DATASET_SLUG,
    'REQUESTED_TRAINING_DATASET_SLUG': REQUESTED_TRAINING_DATASET_SLUG,
    'REQUESTED_TEACHER_DATASET_SLUG': REQUESTED_TEACHER_DATASET_SLUG,
    'REQUESTED_WARM_START_DATASET_SLUG': REQUESTED_WARM_START_DATASET_SLUG,
    'SOURCE_DATASET_SLUG': SOURCE_DATASET_SLUG,
    'TRAINING_DATASET_SLUG': TRAINING_DATASET_SLUG,
    'TEACHER_DATASET_SLUG': TEACHER_DATASET_SLUG,
    'WARM_START_DATASET_SLUG': WARM_START_DATASET_SLUG,
    'SOURCE_INPUT_DIR': str(SOURCE_INPUT_DIR),
    'TRAINING_INPUT_DIR': str(TRAINING_INPUT_DIR),
    'TEACHER_INPUT_DIR': str(TEACHER_INPUT_DIR),
    'WARM_START_INPUT_DIR': str(WARM_START_INPUT_DIR),
    'REPO_URL': REPO_URL,
    'BRANCH': BRANCH,
    'WORKSPACE_DIR': str(WORKSPACE_DIR),
    'OUTPUT_DIR': str(OUTPUT_DIR),
    'RESUME_WARM_START_DIR': str(RESUME_WARM_START_DIR),
    'HF_BASE_MODEL_REPO': HF_BASE_MODEL_REPO,
    'HF_BASE_MODEL_REVISION': HF_BASE_MODEL_REVISION,
    'BASE_MODEL': BASE_MODEL,
    'TRAIN_PROFILE': TRAIN_PROFILE,
    'GPU_PROFILE': GPU_PROFILE,
    'LAUNCH_MODE': LAUNCH_MODE,
    'FORCE_RESTART': FORCE_RESTART,
    'PACKAGE_RESUME_BUNDLE': PACKAGE_RESUME_BUNDLE,
    'RUN_BENCHMARK_AFTER_TRAIN': RUN_BENCHMARK_AFTER_TRAIN,
    'ENABLE_SQLITE_DATASET_MANIFEST': ENABLE_SQLITE_DATASET_MANIFEST,
    'ALLOW_GIT_CLONE_FALLBACK': ALLOW_GIT_CLONE_FALLBACK,
    'TRY_REPO_LFS_ASSETS': TRY_REPO_LFS_ASSETS,
}


## Step 1 - Materialize the repo and install dependencies

This notebook first auto-detects attached Kaggle inputs by their contents, because Kaggle mount names do not always match the dataset card name. It prefers a Kaggle dataset attachment that contains the `source/` tree, because that keeps the training path independent from Git LFS and Kaggle network toggles. If no source dataset is detected, it falls back to downloading a GitHub source archive for `Supermix_29` and only extracts `source/`.


In [ ]:
import shutil
import subprocess
import sys

from importlib.metadata import PackageNotFoundError, version


def run(cmd: list[str], cwd: Path | None = None, env: dict[str, str] | None = None) -> None:
    print('+', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True, cwd=str(cwd) if cwd else None, env=env)
    except subprocess.CalledProcessError as exc:
        joined = ' '.join(cmd)
        raise RuntimeError(f'Command failed with exit code {exc.returncode}: {joined}') from exc


def find_source_snapshot_dir(root: Path) -> Path | None:
    candidates = [
        root / 'source',
        root,
    ]
    for candidate in candidates:
        if (candidate / 'qwen_supermix_pipeline.py').exists():
            return candidate
    return None


def extract_archives(input_dir: Path, dest_dir: Path) -> list[Path]:
    extracted: list[Path] = []
    if not input_dir.exists():
        return extracted
    dest_dir.mkdir(parents=True, exist_ok=True)
    for archive_path in sorted(input_dir.rglob('*.zip')):
        print(f'Extracting archive: {archive_path} -> {dest_dir}')
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(dest_dir)
        extracted.append(archive_path)
    return extracted


if WORKSPACE_DIR.exists():
    print(f'Removing existing workspace at {WORKSPACE_DIR} to avoid stale state...')
    shutil.rmtree(WORKSPACE_DIR)

source_snapshot_dir = find_source_snapshot_dir(SOURCE_INPUT_DIR)
if source_snapshot_dir is not None:
    print(f'Using Kaggle source dataset: {source_snapshot_dir}')
    (WORKSPACE_DIR / 'source').parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source_snapshot_dir, WORKSPACE_DIR / 'source', dirs_exist_ok=True)
else:
    source_archives = extract_archives(SOURCE_INPUT_DIR, WORKSPACE_DIR)
    extracted_source_dir = find_source_snapshot_dir(WORKSPACE_DIR)
    if extracted_source_dir is not None:
        if extracted_source_dir == WORKSPACE_DIR:
            raise RuntimeError('Source archive must contain a top-level source/ directory.')
        print(f'Using extracted Kaggle source archive(s): {source_archives}')
    else:
        if not ALLOW_GIT_CLONE_FALLBACK:
            raise FileNotFoundError(
                f'Missing source dataset at {SOURCE_INPUT_DIR}. Attach a Kaggle dataset named {SOURCE_DATASET_SLUG} or enable ALLOW_GIT_CLONE_FALLBACK.'
            )
        repo_name = REPO_URL.rstrip('/').rsplit('/', 1)[-1].removesuffix('.git')
        archive_url = f'https://codeload.github.com/kai9987kai/{repo_name}/zip/refs/heads/{BRANCH}'
        archive_path = KAGGLE_WORKING_ROOT / f'{repo_name}_{BRANCH}_source.zip'
        extract_root = KAGGLE_WORKING_ROOT / '_repo_source_extract'
        if archive_path.exists():
            archive_path.unlink()
        if extract_root.exists():
            shutil.rmtree(extract_root)
        print(f'Using archive fallback instead of git clone: {archive_url}')
        urllib.request.urlretrieve(archive_url, archive_path)
        with zipfile.ZipFile(archive_path) as zf:
            source_prefix = f'{repo_name}-{BRANCH}/source/'
            members = [name for name in zf.namelist() if name.startswith(source_prefix)]
            if not members:
                raise RuntimeError(f'Archive fallback did not contain {source_prefix}')
            zf.extractall(extract_root, members)
        extracted_source_dir = extract_root / f'{repo_name}-{BRANCH}' / 'source'
        shutil.copytree(extracted_source_dir, WORKSPACE_DIR / 'source', dirs_exist_ok=True)
        print(f'Using downloaded source archive fallback: {extracted_source_dir}')

run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'source/requirements_train_build.txt'], cwd=WORKSPACE_DIR)

extra_packages = [
    'transformers',
    'peft',
    'accelerate',
    'safetensors',
    'matplotlib',
    'tokenizers',
    'huggingface_hub',
    'sentencepiece',
    'tqdm',
]
if PINNED_VERSIONS:
    pinned = [f'{name}=={ver}' for name, ver in PINNED_VERSIONS.items()]
    run([sys.executable, '-m', 'pip', 'install', '-q', *pinned], cwd=WORKSPACE_DIR)
else:
    run([sys.executable, '-m', 'pip', 'install', '-q', *extra_packages], cwd=WORKSPACE_DIR)

for package_name in ['torch', 'transformers', 'peft', 'accelerate', 'safetensors']:
    try:
        print(package_name, version(package_name))
    except PackageNotFoundError:
        print(package_name, '<missing>')

import torch
print('torch.cuda.is_available()', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable after dependency install. Restart the Kaggle session and rerun from the top.')


## Step 2 - Optional SQLite dataset manifest

Kaggle does not give this pipeline a built-in training database, but Python includes `sqlite3`. This optional cell creates a writable SQLite manifest under `/kaggle/working` so you can audit the attached training bundle without changing the model architecture.


In [ ]:
import hashlib
import sqlite3
import zipfile
from datetime import datetime, timezone

DATASET_FILE_NAMES = [
    'conversation_data.quality_anchor_v2.jsonl',
    'conversation_data.coding_knowledge_2026_02_19.jsonl',
    'conversation_data.world_events_2026_02_19.jsonl',
    'conversation_data.supermix_plus_v27_500k.jsonl',
    'conversation_data.mega_reasoning_creative_v25_75582.jsonl',
    'conversation_data.mega_creative_250k_v2.jsonl',
]


def extract_training_archives() -> list[Path]:
    extracted: list[Path] = []
    if not TRAINING_INPUT_DIR.exists():
        return extracted
    dest_dir = WORKSPACE_DIR / 'datasets'
    dest_dir.mkdir(parents=True, exist_ok=True)
    for archive_path in sorted(TRAINING_INPUT_DIR.rglob('*.zip')):
        print(f'Extracting training archive: {archive_path} -> {dest_dir}')
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(dest_dir)
        extracted.append(archive_path)
    return extracted


def resolve_dataset_files() -> list[str]:
    resolved: list[str] = []
    missing: list[str] = []
    search_roots = [
        TRAINING_INPUT_DIR,
        TRAINING_INPUT_DIR / 'datasets',
        WORKSPACE_DIR / 'datasets',
        WORKSPACE_DIR,
        KAGGLE_WORKING_ROOT,
    ]
    for name in DATASET_FILE_NAMES:
        candidates: list[Path] = []
        for root in search_roots:
            if not root.exists():
                continue
            direct = [
                root / name,
                root / 'datasets' / name,
            ]
            for candidate in direct:
                if candidate.exists() and candidate.is_file():
                    candidates.append(candidate)
            if not candidates:
                for hit in sorted(root.rglob(name)):
                    if hit.is_file():
                        candidates.append(hit)
        chosen = next((candidate for candidate in candidates if candidate.exists()), None)
        if chosen is None:
            missing.append(name)
        else:
            resolved.append(str(chosen.resolve()))
    if missing:
        formatted = '\n'.join(f'- {name}' for name in missing)
        mounted_inputs = ', '.join(MOUNTED_INPUTS) if MOUNTED_INPUTS else '<none>'
        raise FileNotFoundError(
            'Missing training JSONL files. '
            f'Requested dataset slug: {REQUESTED_TRAINING_DATASET_SLUG}. '
            f'Resolved training input dir: {TRAINING_INPUT_DIR}. '
            f'Mounted inputs: {mounted_inputs}. '
            'Attach a Kaggle dataset containing these files at its root, in a datasets/ folder, or inside an attached zip archive:\n'
            f'{formatted}'
        )
    return resolved


TRAINING_ARCHIVES = extract_training_archives()
if TRAINING_ARCHIVES:
    print('Attached training archive(s):')
    for archive_path in TRAINING_ARCHIVES:
        print('-', archive_path)

DATA_FILES = resolve_dataset_files()
print('Resolved dataset files:')
for dataset_path in DATA_FILES:
    print('-', dataset_path)

if ENABLE_SQLITE_DATASET_MANIFEST:
    SQLITE_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(SQLITE_MANIFEST_PATH)
    conn.execute(
        '''
        CREATE TABLE IF NOT EXISTS dataset_manifest (
            rel_path TEXT PRIMARY KEY,
            row_count INTEGER NOT NULL,
            byte_count INTEGER NOT NULL,
            sha256 TEXT NOT NULL,
            updated_at_utc TEXT NOT NULL
        )
        '''
    )
    manifest_rows = []
    for dataset_path in DATA_FILES:
        abs_path = Path(dataset_path)
        if not abs_path.exists():
            raise FileNotFoundError(f'Missing dataset file: {abs_path}')
        if abs_path.is_relative_to(KAGGLE_INPUT_ROOT):
            rel_path = str(abs_path.relative_to(KAGGLE_INPUT_ROOT))
        elif abs_path.is_relative_to(KAGGLE_WORKING_ROOT):
            rel_path = str(abs_path.relative_to(KAGGLE_WORKING_ROOT))
        else:
            rel_path = str(abs_path)
        sha256 = hashlib.sha256()
        with abs_path.open('rb') as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b''):
                sha256.update(chunk)
        row_count = 0
        with abs_path.open('r', encoding='utf-8', errors='replace') as fh:
            for line in fh:
                if line.strip():
                    row_count += 1
        byte_count = abs_path.stat().st_size
        updated_at_utc = datetime.now(timezone.utc).isoformat()
        conn.execute(
            'REPLACE INTO dataset_manifest(rel_path, row_count, byte_count, sha256, updated_at_utc) VALUES (?, ?, ?, ?, ?)',
            (rel_path, row_count, byte_count, sha256.hexdigest(), updated_at_utc),
        )
        manifest_rows.append((rel_path, row_count, byte_count))
    conn.commit()
    print('SQLite manifest:', SQLITE_MANIFEST_PATH)
    for rel_path, row_count, byte_count in manifest_rows:
        print(f'- {rel_path}: rows={row_count} bytes={byte_count}')
    conn.close()
else:
    print('SQLite dataset manifest disabled.')


## Step 3 - Build the current training command

This cell keeps the same architecture because it still launches `source/qwen_supermix_pipeline.py`. It exposes the current pipeline features such as grouped eval splitting, true SFT packing, newer distillation ranking controls, preference verbosity controls, and optional self-play knobs. It also auto-detects common Kaggle GPU tiers and only adjusts memory and throughput knobs, not the model family. Teacher distillation and warm-start are enabled automatically when you attach the matching Kaggle datasets.


In [ ]:
import json
import os
import shlex
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone

from huggingface_hub import snapshot_download

LOGICAL_CPU = max(1, os.cpu_count() or 1)
INTEROP_CPU = max(1, min(4, LOGICAL_CPU // 2))


def is_pid_running(pid: int | None) -> bool:
    if pid is None or pid <= 0:
        return False
    try:
        os.kill(pid, 0)
    except OSError:
        return False
    return True


def get_running_train_pid() -> int | None:
    if not PROCESS_PID_PATH.exists():
        return None
    try:
        pid = int(PROCESS_PID_PATH.read_text(encoding='utf-8').strip())
    except Exception:
        return None
    if is_pid_running(pid):
        return pid
    return None


def get_latest_adapter_checkpoint(run_dir: Path | None) -> Path | None:
    if run_dir is None or not run_dir.exists():
        return None
    direct_adapter = run_dir / 'adapter'
    if (direct_adapter / 'adapter_config.json').exists() and (direct_adapter / 'adapter_model.safetensors').exists():
        return direct_adapter.resolve()
    if (run_dir / 'adapter_config.json').exists() and (run_dir / 'adapter_model.safetensors').exists():
        return run_dir.resolve()
    latest_file = run_dir / 'latest_adapter_checkpoint.txt'
    if latest_file.exists():
        checkpoint_dir = latest_file.read_text(encoding='utf-8').strip()
        if checkpoint_dir:
            raw = Path(checkpoint_dir)
            candidates = [raw]
            if not raw.is_absolute():
                candidates.append(WORKSPACE_DIR / raw)
                candidates.append(run_dir / raw)
            for candidate in candidates:
                if candidate.exists():
                    return candidate.resolve()
    checkpoints_dir = run_dir / 'checkpoints'
    if not checkpoints_dir.exists():
        return None
    metas = sorted(checkpoints_dir.rglob('checkpoint_meta.json'), key=lambda path: path.stat().st_mtime, reverse=True)
    for meta in metas:
        adapter_dir = meta.parent / 'adapter'
        if adapter_dir.exists():
            return adapter_dir.resolve()
    return None


def extract_warm_start_archives() -> Path | None:
    if not WARM_START_INPUT_DIR.exists():
        return None
    archive_paths = sorted(WARM_START_INPUT_DIR.rglob('*.zip'))
    if not archive_paths:
        return None
    extracted_dir = KAGGLE_WORKING_ROOT / 'attached_warm_start_input'
    if extracted_dir.exists():
        shutil.rmtree(extracted_dir)
    extracted_dir.mkdir(parents=True, exist_ok=True)
    for archive_path in archive_paths:
        print(f'Extracting warm-start archive: {archive_path} -> {extracted_dir}')
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(extracted_dir)
    return extracted_dir


def find_attached_warm_start_dir() -> Path | None:
    candidates = [
        WARM_START_INPUT_DIR,
        WARM_START_INPUT_DIR / 'artifacts',
    ]
    for candidate in candidates:
        if candidate.exists() and get_latest_adapter_checkpoint(candidate) is not None:
            return candidate
    if WARM_START_INPUT_DIR.exists():
        for child in sorted(WARM_START_INPUT_DIR.iterdir()):
            if child.is_dir() and get_latest_adapter_checkpoint(child) is not None:
                return child
    extracted_dir = extract_warm_start_archives()
    if extracted_dir is not None and get_latest_adapter_checkpoint(extracted_dir) is not None:
        return extracted_dir
    if extracted_dir is not None:
        for child in sorted(extracted_dir.iterdir()):
            if child.is_dir() and get_latest_adapter_checkpoint(child) is not None:
                return child
    return None


INPUT_WARM_START_DIR = find_attached_warm_start_dir()


def seed_working_warm_start() -> Path | None:
    existing_checkpoint = get_latest_adapter_checkpoint(RESUME_WARM_START_DIR)
    if existing_checkpoint is not None:
        return existing_checkpoint
    input_checkpoint = get_latest_adapter_checkpoint(INPUT_WARM_START_DIR)
    if input_checkpoint is None or INPUT_WARM_START_DIR is None:
        return None
    print(f'Seeding writable warm-start artifact from Kaggle input: {INPUT_WARM_START_DIR}')
    shutil.copytree(INPUT_WARM_START_DIR, RESUME_WARM_START_DIR, dirs_exist_ok=True)
    return get_latest_adapter_checkpoint(RESUME_WARM_START_DIR)


def read_warm_start_resume_metadata(run_dir: Path | None) -> dict[str, float]:
    if run_dir is None:
        return {}
    root = Path(run_dir)
    candidates = [
        root / 'benchmark_results.json',
        root.parent / 'benchmark_results.json',
        root.parent / 'checkpoint_meta.json',
        root / 'checkpoint_meta.json',
    ]
    for candidate in candidates:
        if not candidate.exists():
            continue
        try:
            raw = json.loads(candidate.read_text(encoding='utf-8'))
        except Exception:
            continue
        train_stats = raw.get('train_stats', raw) if isinstance(raw, dict) else {}
        if not isinstance(train_stats, dict):
            continue
        return {
            'resume_sft_steps': int(float(train_stats.get('sft_steps', 0.0) or 0.0)),
            'resume_preference_steps': int(float(train_stats.get('preference_steps', 0.0) or 0.0)),
            'resume_sft_loss_mean': float(train_stats.get('sft_loss_mean', 0.0) or 0.0),
            'resume_preference_loss_mean': float(train_stats.get('preference_loss_mean', 0.0) or 0.0),
        }
    return {}


def append_scalar_arg(cmd: list[str], flag: str, value: object) -> None:
    if value is None:
        return
    cmd.extend([flag, str(value)])


def append_bool_flag(cmd: list[str], flag: str, enabled: bool) -> None:
    if enabled:
        cmd.append(flag)


def detect_gpu_profile() -> tuple[str, str, int]:
    query = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
        text=True,
    ).splitlines()[0]
    gpu_name_raw, memory_mb_raw = [part.strip() for part in query.split(',', 1)]
    gpu_name = gpu_name_raw.upper()
    memory_mb = int(float(memory_mb_raw))
    forced = str(GPU_PROFILE).strip().lower()
    if forced and forced != 'auto':
        return forced, gpu_name_raw, memory_mb
    if 'A100' in gpu_name:
        return 'a100', gpu_name_raw, memory_mb
    if 'L4' in gpu_name:
        return 'l4', gpu_name_raw, memory_mb
    if 'P100' in gpu_name:
        return 'p100', gpu_name_raw, memory_mb
    if 'T4' in gpu_name:
        return 't4', gpu_name_raw, memory_mb
    return 'generic', gpu_name_raw, memory_mb


def extract_teacher_archives() -> list[Path]:
    extracted: list[Path] = []
    if not TEACHER_INPUT_DIR.exists():
        return extracted
    dest_dir = WORKSPACE_DIR / 'runtime_python'
    dest_dir.mkdir(parents=True, exist_ok=True)
    for archive_path in sorted(TEACHER_INPUT_DIR.rglob('*.zip')):
        print(f'Extracting teacher archive: {archive_path} -> {dest_dir}')
        with zipfile.ZipFile(archive_path) as zf:
            zf.extractall(dest_dir)
        extracted.append(archive_path)
    return extracted


def resolve_teacher_asset(filename: str) -> Path | None:
    candidates = [
        TEACHER_INPUT_DIR / filename,
        TEACHER_INPUT_DIR / 'runtime_python' / filename,
        WORKSPACE_DIR / 'runtime_python' / filename,
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return None


TEACHER_ARCHIVES = extract_teacher_archives()
if TEACHER_ARCHIVES:
    print('Attached teacher archive(s):')
    for archive_path in TEACHER_ARCHIVES:
        print('-', archive_path)


def ensure_base_model_dir() -> str:
    required_files = [
        BASE_MODEL_DIR / 'config.json',
        BASE_MODEL_DIR / 'tokenizer_config.json',
    ]
    weight_files = list(BASE_MODEL_DIR.glob('*.safetensors')) + list(BASE_MODEL_DIR.glob('pytorch_model*.bin'))
    complete_sentinel = BASE_MODEL_DIR / '.complete'
    if all(path.exists() for path in required_files) and weight_files and complete_sentinel.exists():
        print(f'Using cached base model directory: {BASE_MODEL_DIR}')
        return str(BASE_MODEL_DIR)
    tmp_dir = BASE_MODEL_DIR.with_name(BASE_MODEL_DIR.name + '.tmp')
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    if BASE_MODEL_DIR.exists():
        shutil.rmtree(BASE_MODEL_DIR)
    print(
        f'Downloading base model snapshot: {HF_BASE_MODEL_REPO}@{HF_BASE_MODEL_REVISION} -> {BASE_MODEL_DIR}'
    )
    try:
        snapshot_download(
            repo_id=HF_BASE_MODEL_REPO,
            revision=HF_BASE_MODEL_REVISION,
            local_dir=str(tmp_dir),
            cache_dir=str(HF_HOME / 'hub'),
        )
        downloaded_weights = list(tmp_dir.glob('*.safetensors')) + list(tmp_dir.glob('pytorch_model*.bin'))
        if not all(path.exists() for path in [tmp_dir / 'config.json', tmp_dir / 'tokenizer_config.json']) or not downloaded_weights:
            raise RuntimeError('Downloaded base model snapshot is incomplete.')
        tmp_complete = tmp_dir / '.complete'
        tmp_complete.write_text(HF_BASE_MODEL_REVISION, encoding='utf-8')
        tmp_dir.rename(BASE_MODEL_DIR)
    except Exception as exc:
        shutil.rmtree(tmp_dir, ignore_errors=True)
        raise RuntimeError(
            'Failed to download the base model snapshot from Hugging Face. Ensure Kaggle Internet is enabled and rerun this cell.'
        ) from exc
    return str(BASE_MODEL_DIR)


BASE_MODEL = ensure_base_model_dir()

gpu_profile, detected_gpu_name, detected_gpu_memory_mb = detect_gpu_profile()
print(f'GPU profile: {gpu_profile} ({detected_gpu_name}, {detected_gpu_memory_mb} MiB)')

base_scalar_args = {
    '--max_records': 600000,
    '--max_source_fraction': 0.52,
    '--max_synthetic_fraction': 0.06,
    '--max_prompt_signature_count': 4,
    '--data_log_every_records': 2000,
    '--prompt_signature_cap_exempt_sources': 'conversation_data.quality_anchor_v2.jsonl,conversation_data.mega_reasoning_creative_v25_75582.jsonl',
    '--eval_size': 500,
    '--eval_split_mode': 'auto',
    '--eval_min_quality_score': 1.05,
    '--max_length': 448,
    '--batch_size': 1,
    '--grad_accum_steps': 16,
    '--epochs': 6,
    '--max_steps': 6200,
    '--lr': '1.0e-5',
    '--sft_lr_schedule': 'cosine_restarts',
    '--sft_lr_restart_period': 620,
    '--sft_warmup_steps': 30,
    '--sft_min_lr_ratio': 0.22,
    '--sft_max_grad_norm': 0.9,
    '--sft_focal_gamma': 1.35,
    '--sft_eval_every_steps': 240,
    '--sft_early_stop_patience': 5,
    '--sft_curriculum_quality_ramp': 0.22,
    '--sft_grad_noise_eta': 0.01,
    '--train_log_every_steps': 1,
    '--save_every_steps': SAVE_EVERY_STEPS,
    '--weight_decay': 0.02,
    '--lora_r': 32,
    '--lora_alpha': 64,
    '--lora_dropout': 0.03,
    '--lora_init': 'pissa_niter_4',
    '--lora_plus_ratio': 16,
    '--neftune_noise_alpha': 5.0,
    '--sft_weight_mode': 'quality',
    '--sft_min_weight': 0.62,
    '--sft_max_weight': 1.88,
    '--sft_synthetic_prompt_weight': 0.62,
    '--sft_teacher_source_weight': 0.92,
    '--sft_quality_anchor_boost': 1.14,
    '--sft_coding_boost': 1.24,
    '--sft_events_boost': 1.08,
    '--sft_reasoning_boost': 1.28,
    '--sft_prompt_skill_boost': 1.17,
    '--sft_conversation_boost': 1.24,
    '--sft_creativity_boost': 1.16,
    '--sft_knowledge_density_boost': 1.22,
    '--sft_rdrop_alpha': 0.05,
    '--sft_length_bucket_window_mult': 24,
    '--sft_followup_paraphrase_aug': 1,
    '--sft_followup_paraphrase_weight': 0.68,
    '--sft_min_quality_score': 0.98,
    '--sft_quality_filter_exempt_sources': 'conversation_data.quality_anchor_v2.jsonl,conversation_data.world_events_2026_02_19.jsonl',
    '--sft_source_balance_strength': 0.66,
    '--sft_source_balance_max_scale': 1.95,
    '--sft_packing_max_samples_per_row': 3,
    '--sft_selection_strategy': 'none',
    '--sft_selection_keep_ratio': 1.0,
    '--sft_selection_min_keep': 0,
    '--sft_selection_max_keep': 0,
    '--sft_selection_hardness_target': 0.45,
    '--sft_selection_hardness_bandwidth': 0.20,
    '--sft_selection_budget_mode': 'tokens',
    '--sft_selection_budget_power': 0.5,
    '--sft_selection_scope': 'all',
    '--sft_selection_scope_min_words': 8,
    '--preference_objective': 'ipo',
    '--preference_steps': 1500,
    '--preference_rescore_every': 25,
    '--preference_pairs': 34000,
    '--preference_candidate_count': 8,
    '--preference_reject_similarity_min': 0.16,
    '--preference_beta': 1.9,
    '--preference_beta_end': 3.6,
    '--preference_margin': 0.00,
    '--preference_margin_end': 0.00,
    '--preference_label_smoothing': 0.03,
    '--preference_sft_weight': 0.32,
    '--preference_length_weight': 0.08,
    '--preference_length_control_strength': 0.0,
    '--preference_length_control_target_ratio': 1.0,
    '--preference_length_control_max_penalty': 0.0,
    '--preference_hardness_gamma': 1.15,
    '--preference_robust_alpha': 0.30,
    '--preference_robust_eta': 0.08,
    '--preference_robust_clip': 2.5,
    '--preference_wpo_alpha': 0.35,
    '--preference_wpo_clip': 2.5,
    '--preference_reference_anchor_weight': 0.04,
    '--preference_reference_anchor_batch_size': 2,
    '--preference_short_reject_boost': 0.75,
    '--preference_long_reject_boost': 0.25,
    '--preference_min_chosen_quality': 0.92,
    '--preference_min_chosen_words': 8,
    '--preference_min_quality_gap': 0.05,
    '--preference_max_pairs_per_user': 2,
    '--preference_max_pairs_per_source': 360,
    '--preference_mining_mode': 'auto',
    '--preference_mining_progress_every': 30,
    '--preference_mining_max_seconds': 4500,
    '--preference_mining_max_attempt_factor': 20,
    '--preference_coding_focus_boost': 1.30,
    '--preference_reasoning_focus_boost': 1.32,
    '--preference_counterfactual_rejects_per_prompt': 4,
    '--preference_selection_strategy': 'innovation_mix',
    '--preference_selection_keep_ratio': 0.62,
    '--preference_selection_min_keep': 1800,
    '--preference_selection_max_keep': 2400,
    '--preference_selection_hardness_target': 0.46,
    '--preference_selection_hardness_bandwidth': 0.22,
    '--preference_length_bucket_window_mult': 24,
    '--preference_lr': '1.4e-5',
    '--preference_lr_schedule': 'cosine',
    '--preference_warmup_steps': 18,
    '--preference_min_lr_ratio': 0.30,
    '--preference_max_grad_norm': 0.9,
    '--preference_max_new_tokens': 112,
    '--preference_prompt_max_tokens': 352,
    '--preference_self_play_budget': 0,
    '--preference_self_play_curriculum': 'easy_to_hard',
    '--preference_self_play_max_new_tokens': 0,
    '--supermix_distill_ratio': 0.14,
    '--supermix_distill_max': 8000,
    '--supermix_distill_best_of': 3,
    '--supermix_distill_log_every': 40,
    '--supermix_distill_max_seconds': 12000,
    '--supermix_distill_min_quality': 0.93,
    '--supermix_distill_min_gain': 0.18,
    '--supermix_distill_density_bias': 0.20,
    '--supermix_distill_gain_bias': 0.0,
    '--supermix_distill_compactness_bias': 0.0,
    '--supermix_distill_rank_margin': 0.0,
    '--seed': 48,
    '--device_preference': 'cuda,npu,xpu,mps,cpu,dml',
    '--model_dtype': 'auto',
    '--torch_num_threads': LOGICAL_CPU,
    '--torch_interop_threads': INTEROP_CPU,
}

train_profile_scalar_overrides = {
    'fast_free_t4': {
        '--max_records': 260000,
        '--max_prompt_signature_count': 2,
        '--data_log_every_records': 5000,
        '--eval_size': 64,
        '--max_steps': 3200,
        '--sft_eval_every_steps': 0,
        '--sft_early_stop_patience': 0,
        '--save_every_steps': 80,
        '--keep_last_checkpoints': 3,
        '--sft_followup_paraphrase_aug': 0,
        '--sft_selection_strategy': 'utility_topk',
        '--sft_selection_keep_ratio': 0.60,
        '--sft_selection_min_keep': 22000,
        '--sft_selection_max_keep': 36000,
        '--sft_selection_budget_mode': 'tokens',
        '--sft_selection_budget_power': 0.60,
        '--preference_objective': 'none',
        '--preference_pairs': 0,
        '--preference_steps': 0,
        '--preference_candidate_count': 0,
        '--preference_selection_strategy': 'none',
        '--preference_selection_keep_ratio': 1.0,
        '--preference_selection_min_keep': 0,
        '--preference_selection_max_keep': 0,
        '--preference_mining_max_seconds': 0,
        '--supermix_distill_ratio': 0.0,
        '--supermix_distill_max': 0,
    },
    'current_kaggle': {},
    'parity_v28': {},
}
profile_overrides = train_profile_scalar_overrides.get(TRAIN_PROFILE)
if profile_overrides is None:
    raise ValueError(f'Unsupported TRAIN_PROFILE: {TRAIN_PROFILE}')
if profile_overrides:
    print('Applying train-profile scalar overrides:', profile_overrides)
    base_scalar_args.update(profile_overrides)

gpu_profile_scalar_overrides = {
    't4': {
        '--max_length': 384,
        '--grad_accum_steps': 24,
        '--preference_pairs': 24000,
        '--preference_candidate_count': 6,
        '--supermix_distill_max': 5000,
        '--sft_packing_max_samples_per_row': 2,
    },
    'p100': {
        '--max_length': 384,
        '--grad_accum_steps': 28,
        '--preference_pairs': 22000,
        '--preference_candidate_count': 6,
        '--supermix_distill_max': 4000,
        '--sft_packing_max_samples_per_row': 2,
    },
    'l4': {
        '--max_length': 448,
        '--grad_accum_steps': 16,
        '--preference_pairs': 34000,
        '--preference_candidate_count': 8,
        '--supermix_distill_max': 8000,
        '--sft_packing_max_samples_per_row': 3,
    },
    'a100': {
        '--max_length': 512,
        '--grad_accum_steps': 12,
        '--preference_pairs': 42000,
        '--preference_candidate_count': 10,
        '--supermix_distill_max': 12000,
        '--sft_packing_max_samples_per_row': 4,
    },
    'generic': {},
}
profile_scalar_overrides = gpu_profile_scalar_overrides.get(gpu_profile, {})
if profile_scalar_overrides:
    print('Applying GPU-profile scalar overrides:', profile_scalar_overrides)
    base_scalar_args.update(profile_scalar_overrides)

teacher_weights_path = resolve_teacher_asset('champion_model_chat_supermix_v27_500k_ft.pth')
teacher_meta_path = resolve_teacher_asset('chat_model_meta_supermix_v27_500k.json')
if teacher_weights_path is None or teacher_meta_path is None:
    if float(base_scalar_args.get('--supermix_distill_ratio', 0.0) or 0.0) > 0.0:
        print('Supermix teacher assets not found; disabling teacher distillation for this Kaggle run.')
    teacher_weights_path = None
    teacher_meta_path = None
    base_scalar_args['--supermix_distill_ratio'] = 0.0
else:
    print(f'Using Supermix teacher weights: {teacher_weights_path}')
    print(f'Using Supermix teacher meta: {teacher_meta_path}')

base_bool_flags = {
    '--eval_drop_synthetic_prompts': True,
    '--use_rslora': True,
    '--use_dora': True,
    '--sft_length_bucketed_batches': True,
    '--sft_drop_synthetic_prompts': True,
    '--sft_auto_balance_sources': True,
    '--sft_true_packing': TRAIN_PROFILE in {'fast_free_t4', 'current_kaggle'},
    '--preference_allow_template_prompts': True,
    '--preference_length_bucketed_batches': True,
    '--gradient_checkpointing': True,
}

optional_bool_flags = {
    '--supermix_distill_allow_synthetic_prompts': False,
}


def build_train_command() -> list[str]:
    cmd = [
        sys.executable,
        '-u',
        'source/qwen_supermix_pipeline.py',
        '--data',
        *DATA_FILES,
        '--base_model', BASE_MODEL,
        '--output_dir', str(OUTPUT_DIR),
        '--device', DEVICE,
    ]
    for flag, value in base_scalar_args.items():
        append_scalar_arg(cmd, flag, value)
    for flag, enabled in base_bool_flags.items():
        append_bool_flag(cmd, flag, enabled)
    for flag, enabled in optional_bool_flags.items():
        append_bool_flag(cmd, flag, enabled)
    if teacher_weights_path is not None:
        append_scalar_arg(cmd, '--supermix_weights', teacher_weights_path)
    if teacher_meta_path is not None:
        append_scalar_arg(cmd, '--supermix_meta', teacher_meta_path)

    if RUN_BENCHMARK_AFTER_TRAIN:
        append_scalar_arg(cmd, '--benchmark_eval_limit', BENCHMARK_EVAL_LIMIT)
    else:
        cmd.append('--skip_benchmark')

    latest_output_checkpoint = get_latest_adapter_checkpoint(OUTPUT_DIR)
    if latest_output_checkpoint is not None:
        print(f'Resuming from latest checkpoint in {OUTPUT_DIR}')
        cmd.append('--resume_from_latest_checkpoint')
    else:
        warm_start_checkpoint = seed_working_warm_start()
        if warm_start_checkpoint is not None:
            print(f'Warm-starting from checkpoint in {RESUME_WARM_START_DIR}')
            cmd.extend(['--init_adapter_dir', str(warm_start_checkpoint), '--init_adapter_match_lora'])
            warm_start_resume = read_warm_start_resume_metadata(RESUME_WARM_START_DIR)
            for resume_flag, resume_value in warm_start_resume.items():
                if float(resume_value) > 0.0:
                    append_scalar_arg(cmd, f'--{resume_flag}', resume_value)
            if warm_start_resume:
                print('Warm-start resume metadata:', warm_start_resume)
        else:
            print('No Kaggle warm-start artifact found. Starting a fresh LoRA run from the base model.')
    if EXTRA_ARGS:
        print('Applying extra args:', EXTRA_ARGS)
        cmd.extend(EXTRA_ARGS)
    return cmd


TRAIN_CMD = build_train_command()
LAUNCH_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_LOG = LOG_DIR / f'train_{OUTPUT_DIR.name}_{LAUNCH_ID}.out.log'

launch_state = {
    'updated_at_utc': datetime.now(timezone.utc).isoformat(),
    'repo_url': REPO_URL,
    'branch': BRANCH,
    'workspace_dir': str(WORKSPACE_DIR),
    'output_dir': str(OUTPUT_DIR),
    'resume_warm_start_dir': str(RESUME_WARM_START_DIR),
    'source_dataset_slug': SOURCE_DATASET_SLUG,
    'training_dataset_slug': TRAINING_DATASET_SLUG,
    'teacher_dataset_slug': TEACHER_DATASET_SLUG,
    'warm_start_dataset_slug': WARM_START_DATASET_SLUG,
    'input_warm_start_dir': str(INPUT_WARM_START_DIR) if INPUT_WARM_START_DIR is not None else '',
    'base_model': BASE_MODEL,
    'device': DEVICE,
    'gpu_profile': gpu_profile,
    'detected_gpu_name': detected_gpu_name,
    'detected_gpu_memory_mb': detected_gpu_memory_mb,
    'launch_id': LAUNCH_ID,
    'launch_mode': LAUNCH_MODE,
    'pid': None,
    'started_at_utc': '',
    'completed_at_utc': '',
    'return_code': None,
    'out_log': str(OUT_LOG),
    'train_profile': TRAIN_PROFILE,
    'run_benchmark_after_train': RUN_BENCHMARK_AFTER_TRAIN,
    'teacher_weights_path': str(teacher_weights_path) if teacher_weights_path is not None else '',
    'teacher_meta_path': str(teacher_meta_path) if teacher_meta_path is not None else '',
    'command': TRAIN_CMD,
}
STATE_PATH.write_text(json.dumps(launch_state, indent=2), encoding='utf-8')
print('Command preview:')
print(' '.join(shlex.quote(part) for part in TRAIN_CMD))
print(json.dumps(launch_state, indent=2))


## Step 4 - Launch training

This cell launches training and writes logs under `/kaggle/working`. By default it uses `LAUNCH_MODE = 'background'`, so the kernel stays usable and Step 5 can inspect status without guessing from a blocked cell. After an interruption, rerun the config cell above and then rerun this one. The pipeline will resume from the latest checkpoint under `OUTPUT_DIR` as long as the checkpoint files are still present in the current Kaggle session or in an attached warm-start dataset.


In [ ]:
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

running_pid = get_running_train_pid()
if running_pid is not None:
    if not FORCE_RESTART:
        raise RuntimeError(
            f'Training is already running with pid={running_pid}. Use Step 5 to inspect status, or set FORCE_RESTART = True before rerunning this cell.'
        )
    print(f'Force-stopping existing training pid={running_pid}')
    os.kill(running_pid, 15)
    PROCESS_PID_PATH.unlink(missing_ok=True)

print('Launch mode:', LAUNCH_MODE)
print('Log file:', OUT_LOG)

if LAUNCH_MODE == 'background':
    with OUT_LOG.open('a', encoding='utf-8') as log_file:
        process = subprocess.Popen(
            TRAIN_CMD,
            cwd=str(WORKSPACE_DIR),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            stdin=subprocess.DEVNULL,
            text=True,
            env=env,
            start_new_session=True,
        )
    PROCESS_PID_PATH.write_text(str(process.pid), encoding='utf-8')
    launch_state['pid'] = int(process.pid)
    launch_state['started_at_utc'] = datetime.now(timezone.utc).isoformat()
    STATE_PATH.write_text(json.dumps(launch_state, indent=2), encoding='utf-8')
    print(f'Launched background training pid={process.pid}')
    print('Use Step 5 to inspect status, tail logs, and optionally package a resume bundle.')
elif LAUNCH_MODE == 'stream':
    print('Streaming to notebook and', OUT_LOG)
    with OUT_LOG.open('a', encoding='utf-8') as log_file:
        process = subprocess.Popen(
            TRAIN_CMD,
            cwd=str(WORKSPACE_DIR),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=env,
        )
        PROCESS_PID_PATH.write_text(str(process.pid), encoding='utf-8')
        launch_state['pid'] = int(process.pid)
        launch_state['started_at_utc'] = datetime.now(timezone.utc).isoformat()
        STATE_PATH.write_text(json.dumps(launch_state, indent=2), encoding='utf-8')
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end='')
            log_file.write(line)
            log_file.flush()
        return_code = process.wait()
    launch_state['completed_at_utc'] = datetime.now(timezone.utc).isoformat()
    launch_state['return_code'] = int(return_code)
    STATE_PATH.write_text(json.dumps(launch_state, indent=2), encoding='utf-8')
    PROCESS_PID_PATH.unlink(missing_ok=True)
    if return_code != 0:
        raise RuntimeError(f'Training exited with code {return_code}')
    print('Training finished successfully.')
else:
    raise ValueError(f'Unsupported LAUNCH_MODE: {LAUNCH_MODE}')


## Step 5 - Reattach, inspect progress, and package a Kaggle resume bundle


In [ ]:
from collections import deque
import json
import tarfile

latest_checkpoint = get_latest_adapter_checkpoint(OUTPUT_DIR)
print('Latest checkpoint:', latest_checkpoint)

launch_state = {}
if STATE_PATH.exists():
    launch_state = json.loads(STATE_PATH.read_text(encoding='utf-8'))
    print(
        'Launch runtime:',
        launch_state.get('train_profile'),
        launch_state.get('gpu_profile'),
        launch_state.get('detected_gpu_name'),
    )
else:
    print('No saved launch state yet.')

launch_pid = launch_state.get('pid')
if isinstance(launch_pid, int):
    launch_pid_running = is_pid_running(launch_pid)
    print('Launch pid:', launch_pid, 'running=' + str(launch_pid_running))
    if not launch_pid_running and PROCESS_PID_PATH.exists():
        PROCESS_PID_PATH.unlink(missing_ok=True)
else:
    print('Launch pid: <missing>')
print('Launch mode:', launch_state.get('launch_mode', LAUNCH_MODE))
print('Started:', launch_state.get('started_at_utc', ''))
print('Completed:', launch_state.get('completed_at_utc', ''))
print('Return code:', launch_state.get('return_code'))

try:
    gpu_status = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total', '--format=csv,noheader,nounits'],
        text=True,
    ).splitlines()
    print('GPU status:')
    for idx, line in enumerate(gpu_status):
        util, used, total = [part.strip() for part in line.split(',')]
        print(f' - gpu{idx}: util={util}% mem={used}/{total} MiB')
except Exception as exc:
    print('GPU status unavailable:', exc)

checkpoints_dir = OUTPUT_DIR / 'checkpoints'
if checkpoints_dir.exists():
    recent_checkpoints = sorted(checkpoints_dir.glob('*'), key=lambda path: path.stat().st_mtime, reverse=True)[:5]
    print('Recent checkpoint dirs:')
    for checkpoint_dir in recent_checkpoints:
        print(' -', checkpoint_dir)
else:
    print('No checkpoint directory yet.')

for cache_name in ['prepared_data_cache_meta.json', 'prepared_train_pairs.jsonl', 'prepared_eval_pairs.jsonl', 'teacher_distill_pairs.jsonl']:
    cache_path = OUTPUT_DIR / cache_name
    print(f'{cache_name}:', cache_path if cache_path.exists() else '<missing>')

active_log_path = Path(str(launch_state.get('out_log') or OUT_LOG))
print('Active log:', active_log_path)
if active_log_path.exists():
    with active_log_path.open('r', encoding='utf-8', errors='replace') as fh:
        tail = deque(fh, maxlen=120)
    tail_lines = list(tail)
    stage_lines = [
        line.strip()
        for line in tail_lines
        if line.startswith('[data]') or line.startswith('[train]') or line.startswith('[checkpoint]') or 'Traceback' in line or 'RuntimeError' in line
    ]
    print('Recent stage summary:')
    for line in stage_lines[-12:]:
        print(' -', line)
    print('Recent log tail:')
    print(''.join(tail_lines))
else:
    print('No log file yet.')

if PACKAGE_RESUME_BUNDLE:
    bundle_root = KAGGLE_WORKING_ROOT / 'kaggle_upload_bundles'
    bundle_root.mkdir(parents=True, exist_ok=True)
    bundle_path = bundle_root / f'{OUTPUT_DIR.name}_resume_bundle.tar.gz'
    with tarfile.open(bundle_path, 'w:gz') as tar:
        for item in [OUTPUT_DIR, LOG_DIR, STATE_PATH, SQLITE_MANIFEST_PATH]:
            if item.exists():
                tar.add(item, arcname=str(item.relative_to(KAGGLE_WORKING_ROOT)))
    print('Resume bundle:', bundle_path)
else:
    print('Resume bundle packaging skipped. Set PACKAGE_RESUME_BUNDLE = True to export one.')

benchmark_path = OUTPUT_DIR / 'benchmark_results.json'
if benchmark_path.exists():
    results = json.loads(benchmark_path.read_text(encoding='utf-8'))
    train_stats = results.get('train_stats', {})
    summary = {
        'eval_split_mode': train_stats.get('eval_split_mode'),
        'sft_true_packing': train_stats.get('sft_true_packing'),
        'sft_rows_before_packing': train_stats.get('sft_rows_before_packing'),
        'sft_rows_after_packing': train_stats.get('sft_rows_after_packing'),
        'sft_packed_rows': train_stats.get('sft_packed_rows'),
        'preference_objective': train_stats.get('preference_objective'),
        'teacher_generated': train_stats.get('teacher_generated'),
        'benchmark_sample_summary': results.get('sample_summary', {}),
    }
    print(json.dumps(summary, indent=2))
else:
    print('No benchmark_results.json yet. This is expected when RUN_BENCHMARK_AFTER_TRAIN = False or training has not finished.')


## Step 6 - Package outputs for save version

Kaggle working storage is ephemeral. This cell creates a zip bundle under `/kaggle/working` so you can save a notebook version or reuse the output as a Kaggle dataset in a later run.


In [ ]:
import zipfile

bundle_path = KAGGLE_WORKING_ROOT / f'{OUTPUT_DIR.name}_bundle.zip'
if bundle_path.exists():
    bundle_path.unlink()

with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root_path in [OUTPUT_DIR, LOG_DIR]:
        if not root_path.exists():
            continue
        for file_path in root_path.rglob('*'):
            if file_path.is_file():
                zf.write(file_path, arcname=file_path.relative_to(KAGGLE_WORKING_ROOT))
    for extra_path in [STATE_PATH, SQLITE_MANIFEST_PATH]:
        if extra_path.exists():
            zf.write(extra_path, arcname=extra_path.relative_to(KAGGLE_WORKING_ROOT))

print('Created output bundle:', bundle_path)
